In [ ]:
# 15 — Push the scorecards to Lakebase (read by find_handymen)
# Upserts gold_handyman_performance → maintops.handyman_performance and gold_handyman_feedback →
# maintops.handyman_feedback (INSERT ... ON CONFLICT DO UPDATE), and keeps the headline numbers on
# maintops.handyman_details in line with the overall scorecard.

In [ ]:
%pip install -q "psycopg[binary]>=3.1"

In [ ]:
try:
    dbutils.library.restartPython()
except NameError:
    pass

In [ ]:
import os, sys, time
sys.path.insert(0, os.getcwd())          # pipeline_lib.py lives next to this notebook
import pipeline_lib as pl
from pyspark.sql import functions as F

dbutils.widgets.text("run_id", "manual")
RUN_ID = dbutils.widgets.get("run_id")
pl.ensure_tables(spark)

import psycopg
PG_URL = dbutils.secrets.get("maintops", "lakebase_pg_url")

In [ ]:
started = time.time()
n_perf = pl.upsert_lakebase(PG_URL, spark.table(pl.T_GOLD_PERFORMANCE), "handyman_performance",
                            ["handyman_user_id", "incident_type"], pl.PERFORMANCE_COLUMNS)
n_fb = pl.upsert_lakebase(PG_URL, spark.table(pl.T_GOLD_FEEDBACK), "handyman_feedback",
                          ["handyman_user_id"], pl.FEEDBACK_COLUMNS)

with psycopg.connect(PG_URL) as conn, conn.cursor() as cur:
    cur.execute("SELECT (SELECT count(*) FROM maintops.handyman_performance), "
                "(SELECT count(*) FROM maintops.handyman_feedback)")
    lb_perf, lb_fb = cur.fetchone()

pl.log_step(spark, RUN_ID, "15_sync_lakebase", started, rows_in=n_perf + n_fb, rows_out=lb_perf + lb_fb,
            checks={"performance_in_lakebase": lb_perf >= n_perf, "feedback_in_lakebase": lb_fb >= n_fb})